In [1]:
# Colab / Jupyter setup commands
%cd /content
!rm -rf /content/pydass
!git clone https://github.com/maksimio/pydass.git /content/pydass
%cd /content/pydass
!pip install numpy scipy

/content
Cloning into '/content/pydass'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 94 (delta 44), reused 72 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 27.38 KiB | 1.71 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/pydass


In [2]:
import sys
import numpy as np
from dass import pareto as dass_pareto
from scipy.optimize import linprog

sys.path.append("/content/pydass")

# Real PyDASS methods
from dass import quality_domination, count_domination

In [3]:
class Variant:
    """
    Lightweight wrapper compatible with the PyDASS functions used in dass.py.
    """
    def __init__(self, name, scores):
        self.name = name
        self.scores = list(scores)
        self.linkedTo = set()
        self.nodominated = True


class Importance:
    """
    Wrapper for PyDASS importance structures.
    """
    def __init__(self, positions, importances, importance_coefs=None):
        self.positions = list(positions)
        self.importances = list(importances)
        self.importance_coefs = list(importance_coefs) if importance_coefs is not None else []


class Scale:
    """
    Wrapper for PyDASS scale object.
    """
    def __init__(self, gradeCount):
        self.gradeCount = int(gradeCount)

In [4]:
# ---------------------------------------------------------
# A) E-book example from the uploaded textbook scans
# Table 9.1 / analytical interval case p 355
# Raw columns:
# [price, mass, backlight, fb2, buttons, processor, microSD, WiFi, cover, ppi]
# ---------------------------------------------------------

ebook_names = ["x1", "x2", "x3", "x4", "x5", "x6"]

ebook_raw_matrix = np.array([
    [3800,170,0,0,1, 800,0,1,10,167],
    [6400,206,1,0,0, 800,0,1,10,212],
    [5500,191,0,1,1,1000,1,1, 3,167],
    [7000,198,1,1,1, 800,1,1, 7,212],
    [6200,175,0,0,0, 800,0,1, 6,212],
    [4200,164,0,1,1, 800,1,0, 6,167],
], dtype=float)

ebook_criterion_types = ["min","min","max","max","max","max","max","max","max","max"]


# Ранги важности критериев (1 = самый важный).
# Цена(1) > Процессор(2) > Масса(3) > FB2(4) > Подсветка(5) >
# MicroSD(6) > WiFi(7) > Кнопки(8) > Обложка(9) > PPI(10)
ebook_ranks = [1, 3, 5, 4, 8, 2, 6, 7, 9, 10]

In [5]:
def normalize_matrix(X, types): # for all methods for TVK (B, N, lp) - in quanitiative case (without scaling)
    """
    Normalize all criteria to [0, 1].
    'max' => larger is better
    'min' => smaller is better
    """
    X = np.array(X, dtype=float)
    norm = np.zeros_like(X, dtype=float)

    for j in range(X.shape[1]):
        col = X[:, j]
        cmin = col.min()
        cmax = col.max()
        denom = cmax - cmin if cmax != cmin else 1.0

        if types[j] == "max":
            norm[:, j] = (col - cmin) / denom
        else:
            norm[:, j] = (cmax - col) / denom

    return norm


def build_ranking(names, scores, tol=1e-6):
    """
    Build a stable readable ranking:
    x3 = x6 > x4 = x5 > x2 > x1

    Ties preserve original order.
    """
    scores = np.array(scores, dtype=float)
    indexed = list(enumerate(zip(names, scores)))
    indexed.sort(key=lambda x: (-x[1][1], x[0]))

    sorted_names = [item[1][0] for item in indexed]
    sorted_scores = [item[1][1] for item in indexed]

    groups = [[sorted_names[0]]]
    for i in range(1, len(sorted_scores)):
        if abs(sorted_scores[i] - sorted_scores[i - 1]) <= tol:
            groups[-1].append(sorted_names[i])
        else:
            groups.append([sorted_names[i]])

    return " > ".join(" = ".join(group) for group in groups)


def print_block(title, data):
    """
    Clean unified output block.
    """
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    for k, v in data.items():
        print(f"{k:<14}: {v}")

In [6]:
def weights_generation(n, ranks, delta=0.05, seed=42):
    """
    Генератор весов для SMARTS и интервальных методов (LP, Monte Carlo).
    Параметры:
        n      - число критериев
        ranks  - список рангов (1 = наиважнейший)
        delta  - полуширина интервала вокруг SMARTS-весов
    Возвращает dict с ключами: 'smart', 'smarts', 'intervals'
    Используется только для Hostel. Для Ebook веса берутся из λ (Table 9.2).
    """
    ranks = np.array(ranks, dtype=float)
    assert len(ranks) == n

    # SMARTS: ранговые веса 1/rank, нормированные (стандартная формула)
    raw = 1.0 / ranks
    w_smarts = raw / raw.sum()

    # Интервальные веса: [smarts - delta, smarts + delta]
    w_lo = np.clip(w_smarts - delta, 1e-6, None)
    w_hi = np.clip(w_smarts + delta, None, 1.0)
    # Нормируем центры обратно чтобы SMARTS-веса лежали внутри интервалов
    # Проверка: w_smarts[j] ∈ [w_lo[j], w_hi[j]] для всех j
    consistency_check = all(
        w_lo[j] <= w_smarts[j] <= w_hi[j]
        for j in range(len(w_smarts))
    )
    print(f"  SMARTS ⊂ intervals: {consistency_check}")

    # Проверки непротиворечивости интервалов (обязательное условие для LP)
    assert w_lo.sum() <= 1.0 + 1e-9, \
        f"sum(w_lo)={w_lo.sum():.4f} > 1. Уменьшите delta={delta}."
    assert w_hi.sum() >= 1.0 - 1e-9, \
        f"sum(w_hi)={w_hi.sum():.4f} < 1. Увеличьте delta={delta}."

    return {
        "smarts":    w_smarts.tolist(),
        "intervals": list(zip(w_lo.tolist(), w_hi.tolist())),
    }


# ── Веса для ebook выводятся из λ-интервалов Table 9.2 ───────────
# Бинаризованный диапазон каждого критерия
_anal = ebook_raw_matrix.copy()
_anal[:, 9] = (_anal[:, 9] == 212).astype(float)
_anal[:, 5] = (_anal[:, 5] == 1000).astype(float)
_ranges = np.array([_anal[:,j].max()-_anal[:,j].min() for j in range(10)])

# λ из Table 9.2, стр. 356 учебника.
# Порядок элементов = порядок СТОЛБЦОВ матрицы (cols 1..9),
# то есть НЕ по убыванию важности, а по номеру столбца:
# col1=масса, col2=подсветка, col3=fb2, col4=кнопки,
# col5=процессор, col6=microSD, col7=WiFi, col8=обложка, col9=дисплей(ppi)
# Базовый критерий — цена (col0) — в λ не входит.
_lam_minus = np.array([  5, 1200, 100, 250, 300, 100, 100, 150, 400], dtype=float)
#                      масс подсв  fb2  кнп  проц  sd  wifi  обл  ppi
_lam_plus  = np.array([  8, 1700, 200, 450, 500, 200, 200, 250, 700], dtype=float)
#                      масс подсв  fb2  кнп  проц  sd  wifi  обл  ppi
_lam_mid   = (_lam_minus + _lam_plus) / 2

# Ценовой эквивалент = λ × диапазон_критерия
_pe_mid = np.array([_ranges[0]] + [_lam_mid[k]*_ranges[k+1] for k in range(9)])
_pe_lo  = np.array([_ranges[0]] + [_lam_minus[k]*_ranges[k+1] for k in range(9)])
_pe_hi  = np.array([_ranges[0]] + [_lam_plus[k]*_ranges[k+1] for k in range(9)])

# Нормируем через сумму mid (не через lo/hi отдельно!)
_total = _pe_mid.sum()
W = {
    "smart":     (np.full(10, 1.0/10)).tolist(),
    "smarts":    (_pe_mid / _total).tolist(),
    "intervals": list(zip((_pe_lo/_total).tolist(), (_pe_hi/_total).tolist())),
}
print(f"Веса выведены из λ (Table 9.2). sum(w_lo)={sum(i[0] for i in W['intervals']):.4f}")
print(f"  sum(w_hi)={sum(i[1] for i in W['intervals']):.4f}")
print(f"  SMARTS w_price={W['smarts'][0]:.4f} (наибольший — цена важнейший критерий)")

Веса выведены из λ (Table 9.2). sum(w_lo)=0.8559
  sum(w_hi)=1.1441
  SMARTS w_price=0.3964 (наибольший — цена важнейший критерий)


In [7]:
def run_smart(names, raw_X, criterion_types):
    """
    Classical SMART:
    - normalize raw matrix
    - use equal weights
    - compute weighted sum
    """
    norm = normalize_matrix(raw_X, criterion_types)
    weights = np.ones(norm.shape[1], dtype=float) / norm.shape[1]
    scores = norm @ weights

    return {
        "method": "SMART",
        "ranking": build_ranking(names, scores),
        "weights": weights.tolist(),
        "scores": scores.tolist()
    }

In [8]:
def run_smarts(names, raw_X, criterion_types, weights):
    """
    SMARTS на ebook-данных с ранговыми весами из weights_generation().

    Отличие от SMART: веса пропорциональны 1/rank (неравные),
    а не 1/n (равные). Нормировка — та же min-max.
    """
    norm = normalize_matrix(raw_X, criterion_types)
    w = np.array(weights["smarts"])
    scores = norm @ w
    return {
        "method": "SMARTS",
        "ranking": build_ranking(names, scores),
        "weights": [round(x, 6) for x in w.tolist()],
        "scores":  [round(x, 6) for x in scores.tolist()],
    }

In [9]:
def build_pydass_grade_matrix(raw_X, criterion_types, n_grades=11):
    """
    Единообразное преобразование всех критериев в грейдовую шкалу [0, n_grades-1].

    Алгоритм:
      1. Нормируем через normalize_matrix (учитывает min/max тип критерия)
      2. Масштабируем в [0, n_grades-1] и округляем до целых

    Это корректно для PyDASS: все критерии ОЦЕНКИ АЛЬТЕРНАТИВ!!!??? на одной порядковой шкале.
    """
    norm = normalize_matrix(raw_X, criterion_types)
    graded = np.round(norm * (n_grades - 1)).astype(int)
    return graded

In [10]:
def run_tvk_cascade(names, graded_X, ranks=None, importance_coef=1.25):
    """
    Каскадный TVK согласно main.py из PyDASS:
      Шаг 1: Парето на всех вариантах
      Шаг 2: TVK_qual только на Парето-множестве
      Шаг 3: TVK_quant только на TVK_qual-множестве

    ranks - ранги важности критериев (1=наиважнейший).
            Если None — критерии считаются важными в порядке столбцов.

    Свойство вложенности: pareto_set ⊇ qual_set ⊇ quant_set
    """
    def make_variants(local_names, local_X):
        return [Variant(n, row) for n, row in zip(local_names, local_X)]

    n_crit = graded_X.shape[1]
    scale  = Scale(gradeCount=11)

    # Вычисляем positions из ranks:
    # positions[i] = индекс столбца критерия с рангом (i+1)
    if ranks is not None:
        positions = list(np.argsort(ranks))
    else:
        positions = list(range(n_crit))

    # ── Шаг 1: Парето ────────────────────────────────────────────
    variants = make_variants(names, graded_X)
    dass_pareto(variants)

    pareto_names = [v.name for v in variants if v.nodominated]
    pareto_idx   = [list(names).index(n) for n in pareto_names]
    pareto_X     = graded_X[pareto_idx]

    # ── Шаг 2: TVK_qual на Парето-множестве ──────────────────────
    variants_q   = make_variants(pareto_names, pareto_X)
    importance_q = Importance(
        positions=positions,
        importances=[True] * (n_crit - 1),
    )
    quality_domination(variants_q, importance_q, scale)

    qual_names = [v.name for v in variants_q if v.nodominated]
    qual_idx   = [pareto_names.index(n) for n in qual_names]
    qual_X     = pareto_X[qual_idx]

    # ── Шаг 3: TVK_quant на TVK_qual-множестве ───────────────────
    variants_n   = make_variants(qual_names, qual_X)
    importance_n = Importance(
        positions=positions,
        importances=[True] * (n_crit - 1),
        importance_coefs=[importance_coef] * (n_crit - 1),
    )
    count_domination(variants_n, importance_n, scale)

    quant_names = [v.name for v in variants_n if v.nodominated]

    def make_rank_str(selected, all_names):
        rest = [n for n in all_names if n not in selected]
        s = " = ".join(selected)
        if rest:
            s += " > " + " = ".join(rest)
        return s

    return {
        "method":      "TVK_cascade",
        "pareto_set":  pareto_names,
        "pareto_rank": make_rank_str(pareto_names, names),
        "qual_set":    qual_names,
        "qual_rank":   make_rank_str(qual_names, names),
        "quant_set":   quant_names,
        "quant_rank":  make_rank_str(quant_names, names),
    }

In [11]:
def pareto_front(names, norm_X):
    """
    Вычисляет Парето-оптимальное множество по нормированной матрице.

    Альтернатива i доминирует j по Парето, если:
      norm_X[i, k] >= norm_X[j, k]  для всех k
      norm_X[i, k] >  norm_X[j, k]  хотя бы для одного k

    Возвращает список имён Парето-оптимальных альтернатив.
    """
    m = len(names)
    is_dominated = [False] * m
    for i in range(m):
        for j in range(m):
            if i == j:
                continue
            # Проверяем: j доминирует i?
            if np.all(norm_X[j] >= norm_X[i]) and np.any(norm_X[j] > norm_X[i]):
                is_dominated[i] = True
                break
    return [names[i] for i in range(m) if not is_dominated[i]]

In [12]:
def lp_feasible(A_ge=None, b_ge=None, A_le=None, b_le=None, A_eq=None, b_eq=None, bounds=None):
    """
    Generic feasibility LP solver.
    """
    c = np.zeros(len(bounds), dtype=float)

    A_ub = []
    b_ub = []

    if A_le is not None and b_le is not None:
        A_ub.extend(np.asarray(A_le, dtype=float).tolist())
        b_ub.extend(np.asarray(b_le, dtype=float).tolist())

    if A_ge is not None and b_ge is not None:
        A_ge = np.asarray(A_ge, dtype=float)
        b_ge = np.asarray(b_ge, dtype=float)
        A_ub.extend((-A_ge).tolist())
        b_ub.extend((-b_ge).tolist())

    if len(A_ub) == 0:
        A_ub = None
        b_ub = None
    else:
        A_ub = np.asarray(A_ub, dtype=float)
        b_ub = np.asarray(b_ub, dtype=float)

    if A_eq is not None and b_eq is not None:
        A_eq = np.asarray(A_eq, dtype=float)
        b_eq = np.asarray(b_eq, dtype=float)

    res = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs"
    )

    return bool(res.success), (res.x.tolist() if res.success else None)

In [13]:
def solve_lp_example_9_2():
    """
    Unit test: Пример 9.2 (стр. 350).
    Ожидаемые ответы из учебника:
      consistency_feasible = True
      y P^Lambda z         = True
      y P^Lambda u         = False
    """
    A_ge = [
        [ 6.0, -1.0,  0.0],
        [-2.0,  1.0,  0.0],
        [ 2.0,  0.0, -1.0],
        [-0.5,  0.0,  1.0],
        [ 1.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0],
        [ 0.0,  0.0,  1.0],
    ]
    b_ge = [1, 1, 1, 1, 1, 1, 1]
    bounds_mu = [(None, None), (None, None), (None, None)]

    consistency_feasible, _ = lp_feasible(
        A_ge=A_ge, b_ge=b_ge, bounds=bounds_mu
    )

    A_le_yz = [
        [ 6.0, -2.0,  2.0, -0.5],
        [-1.0,  1.0,  0.0,  0.0],
        [ 0.0,  0.0, -1.0,  1.0],
    ]
    b_le_yz = [-1, 3, -2]
    bounds_v = [(0, None)] * 4

    yz_feasible, _ = lp_feasible(
        A_le=A_le_yz, b_le=b_le_yz, bounds=bounds_v
    )

    A_le_yu = [
        [ 6.0, -2.0,  2.0, -0.5],
        [-1.0,  1.0,  0.0,  0.0],
        [ 0.0,  0.0, -1.0,  1.0],
    ]
    b_le_yu = [1, -1, 8]

    yu_feasible, _ = lp_feasible(
        A_le=A_le_yu, b_le=b_le_yu, bounds=bounds_v
    )

    return {
        "method":                "REFERENCE_LP_9_2",
        "consistency_feasible":  consistency_feasible,
        "y_P_lambda_z":          yz_feasible,
        "y_P_lambda_u":          yu_feasible,
    }

In [14]:
def run_interval_lp(names, norm_X, weight_intervals):
    """
    Интервальный LP: k доминирует i если для ВСЕХ w ∈ Lambda:
      (norm_X[k] - norm_X[i]) · w >= 0

    Проверяем через:
      min (norm_X[k] - norm_X[i]) · w >= 0
      s.t. sum(w) = 1,  lo_j <= w_j <= hi_j
    """
    norm_X = np.array(norm_X, dtype=float)
    m, n = norm_X.shape
    bounds = [(lo, hi) for lo, hi in weight_intervals]

    dominated_by = {name: [] for name in names}

    for i in range(m):
        for k in range(m):
            if i == k:
                continue

            c_diff = norm_X[k] - norm_X[i]

            # Шаг 1: min c_diff · w >= 0? (k не хуже i при любых весах)
            res_min = linprog(
                c=c_diff,
                A_eq=[np.ones(n)],
                b_eq=[1.0],
                bounds=bounds,
                method="highs",
            )

            if not (res_min.success and res_min.fun >= -1e-9):
                continue  # есть веса при которых k хуже i — не доминирует

            # Шаг 2: max c_diff · w > 0? (k строго лучше i хотя бы при одних весах)
            res_max = linprog(
                c=-c_diff,
                A_eq=[np.ones(n)],
                b_eq=[1.0],
                bounds=bounds,
                method="highs",
            )

            if res_max.success and (-res_max.fun) > 1e-9:
                dominated_by[names[i]].append(names[k])

    pareto_lp = [name for name in names if len(dominated_by[name]) == 0]
    win_count  = [m - 1 - len(dominated_by[name]) for name in names]

    return {
        "method":       "Interval_LP",
        "pareto_lp":    pareto_lp,
        "ranking":      build_ranking(names, win_count),
        "dominated_by": {k: v for k, v in dominated_by.items() if v},
    }

In [15]:
def run_monte_carlo(names, norm_X, weight_intervals, n_samples=10_000, seed=42):
    """
    Метод Монте-Карло для оценки устойчивости SMARTS-ранжирования.

    При каждой итерации:
      1. Случайный вектор весов w ~ Dirichlet(1), обрезанный к [lo, hi]
      2. Победитель = argmax V(a_i) = argmax(norm_X @ w)
         — это SMARTS-правило при данном w, НЕ LP-доминирование
      3. Частота побед ≈ вероятность быть лучшим по SMARTS
         при неопределённых весах из Λ

    Интерпретация: если альтернатива побеждает в 100% итераций,
    она является SMARTS-победителем при ЛЮБЫХ допустимых весах.
    Это согласуется с Interval LP: если LP-pareto = {x1}, то
    Monte Carlo тоже даст x1 = 100%.
    """
    rng = np.random.default_rng(seed)
    norm_X = np.array(norm_X, dtype=float)
    m, n = norm_X.shape
    lo = np.array([iv[0] for iv in weight_intervals])
    hi = np.array([iv[1] for iv in weight_intervals])

    win_counts = np.zeros(m, dtype=int)

    for _ in range(n_samples):
        w = rng.dirichlet(np.ones(n))
        w = np.clip(w, lo, hi)
        w /= w.sum()
        winner = int(np.argmax(norm_X @ w))  # SMARTS-правило при данном w
        win_counts[winner] += 1

    probs = win_counts / n_samples
    return {
        "method":        "Monte_Carlo_SMARTS",  # ← переименовали
        "n_samples":     n_samples,
        "win_counts":    dict(zip(names, win_counts.tolist())),
        "probabilities": {n: round(float(p), 4) for n, p in zip(names, probs)},
        "ranking":       build_ranking(names, probs.tolist()),
    }

In [16]:
def run_interval_analytical(names, raw_X, signs, lam_minus, lam_plus):
    """
    Аналитический метод §9.4 (формулы 9.25/9.26, Теорема 9.4).

    Параметры:
        names     - имена альтернатив
        raw_X     - матрица (столбец 0 = базовый критерий)
        signs     - знаки критериев: +1=max, -1=min (длина = число столбцов)
        lam_minus - λ⁻ для небазовых критериев (длина = число столбцов - 1)
        lam_plus  - λ⁺ для небазовых критериев

    Правило (9.24): x^y P^Λ x^z ⟺ l ≥ 0, где
        l = s₁·δ₁ + Σ_{sᵢδᵢ>0} sᵢδᵢλᵢ⁻ + Σ_{sᵢδᵢ<0} sᵢδᵢλᵢ⁺
    """
    X = np.array(raw_X, dtype=float)
    m = len(names)
    signs     = np.array(signs,     dtype=float)
    lam_minus = np.array(lam_minus, dtype=float)
    lam_plus  = np.array(lam_plus,  dtype=float)

    wins   = [0] * m
    losses = [0] * m

    for i in range(m):
        for j in range(m):
            if i == j:
                continue

            delta = X[i] - X[j]

            # Базовый критерий (столбец 0)
            l = signs[0] * delta[0]

            # Небазовые критерии (столбцы 1..n-1)
            for k in range(1, X.shape[1]):
                sd = signs[k] * delta[k]
                if sd > 0:
                    l += sd * lam_minus[k - 1]
                elif sd < 0:
                    l += sd * lam_plus[k - 1]

            if l >= 0 and np.any(np.abs(delta) > 1e-9):
                wins[i]   += 1
                losses[j] += 1

    net    = [wins[i] - losses[i] for i in range(m)]
    pareto = [names[i] for i in range(m) if losses[i] == 0]
    return {
        "method":    "Interval_Analytical",
        "ranking":   build_ranking(names, net),
        "pareto_set": pareto,
        "wins":      wins,
        "losses":    losses,
    }

In [17]:
# Для ebook: бинаризация PPI и процессора (как в аналитическом методе)
ebook_analytical_matrix = ebook_raw_matrix.copy()
ebook_analytical_matrix[:, 9] = (ebook_raw_matrix[:, 9] == 212).astype(float)
ebook_analytical_matrix[:, 5] = (ebook_raw_matrix[:, 5] == 1000).astype(float)

ebook_grade_matrix = build_pydass_grade_matrix(ebook_analytical_matrix, ebook_criterion_types)
print("Grade matrix (0..10 scale):")
print(ebook_grade_matrix)

# Нормированная матрица для LP/MC/SMARTS — из бинаризованных данных
ebook_norm_matrix = normalize_matrix(ebook_analytical_matrix, ebook_criterion_types)


# Парето-фронт (независимый от весов)
pareto_result = pareto_front(ebook_names, ebook_norm_matrix)
print(f"Парето-оптимальное множество: {pareto_result}")

# Аддитивные методы
smart_result  = run_smart(ebook_names, ebook_analytical_matrix, ebook_criterion_types)
smarts_result = run_smarts(ebook_names, ebook_analytical_matrix, ebook_criterion_types, W)

# Данные хостела (Table 5.5, стр. 171)
hostel_names = ["x1", "x2", "x3", "x4", "x5", "x6"]
hostel_raw = np.array([
    [30, 8, 7, 6,  4, 2],
    [35, 6, 9, 6,  8, 1],
    [28, 6, 8, 8, 10, 3],
    [34, 7, 7, 7,  7, 2],
    [30, 5, 8, 9,  5, 2],
    [32,10, 6,10,  6, 1],
], dtype=float)
hostel_types = ["min","max","max","max","max","max"]
hostel_grade_matrix = build_pydass_grade_matrix(hostel_raw, hostel_types)
hostel_smart_result = run_smart(hostel_names, hostel_raw, hostel_types)


# Нормированные данные хостела для SMARTS (Table 5.6, стр. 171)
hostel_X_norm = np.array([
    [ 60.0, 33.3,  0.0,   0.0,  50.0],
    [ 20.0,100.0,  0.0,  66.7,   0.0],
    [ 20.0, 66.7, 50.0, 100.0, 100.0],
    [ 40.0, 33.3, 25.0,  50.0,  50.0],
    [  0.0, 66.7, 75.0,  16.7,  50.0],
    [100.0,  0.0,100.0,  33.3,   0.0],
], dtype=float)
hostel_ranks = [1, 2, 3, 4, 5, 6]
hostel_smarts_w = np.array([0.294118, 0.176471, 0.235294, 0.176471, 0.117647])
hostel_scores = hostel_X_norm @ hostel_smarts_w
hostel_smarts_result = {
    "method":  "SMARTS_hostel_textbook",
    "ranking": build_ranking(hostel_names, hostel_scores),
    "scores":  hostel_scores.tolist(),
}

# TVK каскад: ebook и hostel
# importance_coef=3.0: при таком значении цена весит в 3 раза больше
# процессора, что достаточно чтобы x1 (лучшая цена) вошла в quant_set.
# Обоснование: λ_price_range=3200 >> λ_proc=300-500 руб. (Table 9.2, стр. 356)
tvk_ebook_result = run_tvk_cascade(ebook_names, ebook_grade_matrix,
                                    ranks=ebook_ranks, importance_coef=3.0)
tvk_hostel_result = run_tvk_cascade(hostel_names, hostel_grade_matrix, ranks=hostel_ranks)

# Interval LP на ebook-данных
lp_result = run_interval_lp(ebook_names, ebook_norm_matrix, W["intervals"])

# Monte Carlo
mc_result = run_monte_carlo(ebook_names, ebook_norm_matrix, W["intervals"], n_samples=10_000)

ebook_signs     = [-1, -1, +1, +1, +1, +1, +1, +1, +1, +1]
#                цена масс подсв fb2 кнп проц  sd wifi обл ppi
ebook_lam_minus = [  5, 1200, 100, 250, 300, 100, 100, 150, 400]
#                  масс подсв  fb2  кнп  проц  sd  wifi  обл  ppi
ebook_lam_plus  = [  8, 1700, 200, 450, 500, 200, 200, 250, 700]
#                  масс подсв  fb2  кнп  проц  sd  wifi  обл  ppi
# Примечание: порядок следует порядку столбцов матрицы (cols 1-9),
# а НЕ порядку важности критериев (ebook_ranks).
# В учебнике Table 9.2 критерии перечислены в другом порядке —
# здесь они переставлены под индексы матрицы для корректной работы формулы.

analytical_result = run_interval_analytical(
    ebook_names, ebook_analytical_matrix,
    ebook_signs, ebook_lam_minus, ebook_lam_plus
)

# LP unit test (Пример 9.2, стр. 350)
lp_9_2_result = solve_lp_example_9_2()

# ── Dataset 2: Hostel — интервальная версия ──────────────────────────────────
# Ранги: f1=цена(1), f2=размещение(2), f3=дизайн(3),
#        f4=чистота(4), f5=расположение(5), f6=комфорт(6)

W_hostel = weights_generation(n=6, ranks=hostel_ranks, delta=0.04, seed=42)

hostel_norm_matrix = normalize_matrix(hostel_raw, hostel_types)

lp_hostel_result = run_interval_lp(
    hostel_names, hostel_norm_matrix, W_hostel["intervals"]
)
mc_hostel_result = run_monte_carlo(
    hostel_names, hostel_norm_matrix, W_hostel["intervals"], n_samples=10_000
)

# ── Hostel: аналитический метод ──────────────────────────────────
# Базовый критерий: f1 (цена, min). Небазовые: f2..f6.
# λ_j выводим из SMARTS-весов: λ_j ≈ w_j/w_base * range_base
# где range_base = 35-28 = 7 (диапазон цены), w_base не задан явно,
# поэтому используем отношение весов.

hostel_price_range = hostel_raw[:,0].max() - hostel_raw[:,0].min()  # = 7

# Оценочные λ как ценовые эквиваленты на единицу каждого критерия
hostel_ranges_nonbase = np.array([
    hostel_raw[:,j].max() - hostel_raw[:,j].min()
    for j in range(1,6)
])  # [5,3,4,6,2]

# lambda = w_j/sum(w_nonbase) * price_range / range_j
# Используем δ = 20% от λ как интервал неопределённости
hostel_lam_mid = (hostel_smarts_w / hostel_smarts_w.sum()) * hostel_price_range / hostel_ranges_nonbase
hostel_lam_minus = hostel_lam_mid * 0.8
hostel_lam_plus  = hostel_lam_mid * 1.2

hostel_signs = [-1, +1, +1, +1, +1, +1]

analytical_hostel_result = run_interval_analytical(
    hostel_names, hostel_raw,
    hostel_signs, hostel_lam_minus, hostel_lam_plus
)

Grade matrix (0..10 scale):
[[10  9  0  0 10  0  0 10 10  0]
 [ 2  0 10  0  0  0  0 10 10 10]
 [ 5  4  0 10 10 10 10 10  0  0]
 [ 0  2 10 10 10  0 10 10  6 10]
 [ 2  7  0  0  0  0  0 10  4 10]
 [ 9 10  0 10 10  0 10  0  4  0]]
Парето-оптимальное множество: ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
  SMARTS ⊂ intervals: True


In [18]:
# ── Dataset 4: Preflib ───────────────────────────────────────────
# Читаем параметры напрямую из preflib.json

preflib_names = ["A (200)", "B (209)", "C (218)", "D (227)"]

# Оценки по критериям: Borda, Approval, Simpson, Copeland, Dodgson
preflib_raw = np.array([
    [3.889,  8.5, 392,  30,   1],
    [3.118,  5.2, 356,  14,  52],
    [2.334,  2.1, 298, -11, 123],
    [1.688,  0.3, 154, -32,  34],
], dtype=float)

preflib_types = ["max", "max", "max", "max", "max"]

# ── Из JSON: importance ──────────────────────────────────────────
# positions: [3, 4, 1, 2, 5] — порядок критериев по важности (1-based)
# Переводим в 0-based индексы: [2, 3, 0, 1, 4]
# Смысл: критерий 3 (Simpson, col=2) — самый важный
preflib_positions = [p - 1 for p in [3, 4, 1, 2, 5]]

# importanceCoefs: [10, 5, 18, 12, 15] — коэффициенты N-модели
# используются напрямую в count_domination вместо [coef]*n_crit
preflib_ic = [10, 5, 18, 12, 15]

# gradeCount: 100 — шкала оценок 0..100 (из JSON, не 0..10!)
preflib_grade_count = 100

# relativeImportance: [1.0, 0.8, 0.6, 0.7, 0.9]
# ri[i] < 1.0 означает "строго менее важен" → importances[i] = True
# Все значения < 1.0 → все пары: менее важный < более важный
preflib_importances = [ri < 1.0 for ri in [1.0, 0.8, 0.6, 0.7, 0.9]]
# = [False, True, True, True, True]
# False для первой пары означает что критерии 0 и 1 по важности могут быть равны

# ── Нормировка и grade matrix ────────────────────────────────────
preflib_norm = normalize_matrix(preflib_raw, preflib_types)

# Grade matrix в шкале 0..100 (как задано в gradeCount)
preflib_grade_matrix = np.round(preflib_norm * (preflib_grade_count - 1)).astype(int)

print("Preflib grade matrix (0..100 scale):")
print(preflib_grade_matrix)

# ── TVK CASCADE с параметрами из JSON ───────────────────────────
def run_tvk_preflib(names, graded_X, positions, importances, importance_coefs, grade_count):
    """
    TVK каскад с параметрами напрямую из preflib.json:
    - positions: порядок критериев (0-based)
    - importances: список bool (True = строго менее важен)
    - importance_coefs: коэффициенты N-модели из JSON
    - grade_count: размер шкалы (100 для preflib)
    """
    def make_variants(local_names, local_X):
        return [Variant(n, row) for n, row in zip(local_names, local_X)]

    scale = Scale(gradeCount=grade_count)
    n_crit = graded_X.shape[1]

    # Шаг 1: Парето
    variants = make_variants(names, graded_X)
    dass_pareto(variants)
    pareto_names = [v.name for v in variants if v.nodominated]
    pareto_idx   = [list(names).index(n) for n in pareto_names]
    pareto_X     = graded_X[pareto_idx]

    # Шаг 2: Качественная доминация
    variants_q   = make_variants(pareto_names, pareto_X)
    importance_q = Importance(
        positions=positions,
        importances=importances,  # из relativeImportance JSON
    )
    quality_domination(variants_q, importance_q, scale)
    qual_names = [v.name for v in variants_q if v.nodominated]
    qual_idx   = [pareto_names.index(n) for n in qual_names]
    qual_X     = pareto_X[qual_idx]

    # Шаг 3: Количественная доминация
    variants_n   = make_variants(qual_names, qual_X)
    importance_n = Importance(
        positions=positions,
        importances=importances,
        importance_coefs=importance_coefs,  # [10,5,18,12,15] из JSON
    )
    count_domination(variants_n, importance_n, scale)
    quant_names = [v.name for v in variants_n if v.nodominated]

    def make_rank_str(selected, all_names):
        rest = [n for n in all_names if n not in selected]
        s = " = ".join(selected)
        if rest:
            s += " > " + " = ".join(rest)
        return s

    return {
        "method":      "TVK_cascade",
        "pareto_set":  pareto_names,
        "pareto_rank": make_rank_str(pareto_names, names),
        "qual_set":    qual_names,
        "qual_rank":   make_rank_str(qual_names, names),
        "quant_set":   quant_names,
        "quant_rank":  make_rank_str(quant_names, names),
    }

preflib_tvk_result = run_tvk_preflib(
    preflib_names, preflib_grade_matrix,
    positions=preflib_positions,
    importances=preflib_importances,
    importance_coefs=preflib_ic,
    grade_count=preflib_grade_count,
)

# ── SMARTS: веса из importanceCoefs ─────────────────────────────
# importanceCoefs = [10, 5, 18, 12, 15] — абсолютные важности
# Нормируем: w_j = ic_j / sum(ic)
ic = np.array([10, 5, 18, 12, 15], dtype=float)
w_preflib_smarts = ic / ic.sum()  # [0.167, 0.083, 0.300, 0.200, 0.250]

# Интервалы: ±20% от importanceCoefs (нет явных границ в JSON)
w_preflib_lo = np.clip(w_preflib_smarts * 0.8, 1e-6, None)
w_preflib_hi = w_preflib_smarts * 1.2
w_preflib_lo /= w_preflib_lo.sum()  # нормируем чтобы sum(lo) ≤ 1
w_preflib_hi /= w_preflib_hi.sum()  # нормируем чтобы sum(hi) ≥ 1

W_preflib = {
    "smarts":    w_preflib_smarts.tolist(),
    "intervals": list(zip(
        (w_preflib_smarts * 0.8 / (w_preflib_smarts * 0.8).sum()).tolist(),
        (w_preflib_smarts * 1.2 / (w_preflib_smarts * 1.2).sum()).tolist(),
    ))
}

print(f"\nPrefLib SMARTS weights (из importanceCoefs): {np.round(w_preflib_smarts, 4)}")

# ── Аналитический метод ──────────────────────────────────────────
# Базовый критерий: Borda (col 0, max)
# λ выводим из importanceCoefs в пересчёте на единицу Borda
pf_borda_range = preflib_raw[:,0].max() - preflib_raw[:,0].min()
pf_ranges_nb   = np.array([preflib_raw[:,j].max()-preflib_raw[:,j].min()
                            for j in range(1,5)])
# Веса небазовых: из importanceCoefs[1..4]
w_nb_norm = ic[1:] / ic[1:].sum()
pf_lam_mid    = w_nb_norm * pf_borda_range / pf_ranges_nb
pf_lam_minus  = pf_lam_mid * 0.8
pf_lam_plus   = pf_lam_mid * 1.2
preflib_signs = [+1, +1, +1, +1, +1]

analytical_preflib_result = run_interval_analytical(
    preflib_names, preflib_raw,
    preflib_signs, pf_lam_minus, pf_lam_plus
)

# ── Остальные методы ─────────────────────────────────────────────
preflib_smart_result  = run_smart(preflib_names, preflib_raw, preflib_types)
preflib_smarts_result = run_smarts(preflib_names, preflib_raw, preflib_types, W_preflib)
preflib_lp_result     = run_interval_lp(preflib_names, preflib_norm, W_preflib["intervals"])
preflib_mc_result     = run_monte_carlo(
    preflib_names, preflib_norm, W_preflib["intervals"], n_samples=10_000
)
preflib_pareto = pareto_front(preflib_names, preflib_norm)

print(f"Парето-фронт: {preflib_pareto}")
print_block("TVK CASCADE (preflib JSON)", preflib_tvk_result)
print_block("INTERVAL ANALYTICAL (preflib)", analytical_preflib_result)

Preflib grade matrix (0..100 scale):
[[99 99 99 99  0]
 [64 59 84 73 41]
 [29 22 60 34 99]
 [ 0  0  0  0 27]]

PrefLib SMARTS weights (из importanceCoefs): [0.1667 0.0833 0.3    0.2    0.25  ]
Парето-фронт: ['A (200)', 'B (209)', 'C (218)']

TVK CASCADE (preflib JSON)
method        : TVK_cascade
pareto_set    : ['A (200)', 'B (209)', 'C (218)']
pareto_rank   : A (200) = B (209) = C (218) > D (227)
qual_set      : ['A (200)', 'B (209)', 'C (218)']
qual_rank     : A (200) = B (209) = C (218) > D (227)
quant_set     : ['A (200)']
quant_rank    : A (200) > B (209) = C (218) = D (227)

INTERVAL ANALYTICAL (preflib)
method        : Interval_Analytical
ranking       : A (200) > B (209) > C (218) > D (227)
pareto_set    : ['A (200)']
wins          : [3, 2, 1, 0]
losses        : [0, 1, 2, 3]


In [19]:
# ── N-модель: чувствительность к importance_coef ─────────────────
print("=== N-model sensitivity: importance_coef vs quant_set ===")
for coef in [1.1, 1.25, 1.5, 2.0, 3.0, 5.0, 10.0]:
    result = run_tvk_cascade(ebook_names, ebook_grade_matrix,
                             ranks=ebook_ranks, importance_coef=coef)
    print(f"  coef={coef:.2f}: quant_set={result['quant_set']}")

=== N-model sensitivity: importance_coef vs quant_set ===
  coef=1.10: quant_set=['x3', 'x4', 'x6']
  coef=1.25: quant_set=['x3', 'x4', 'x6']
  coef=1.50: quant_set=['x1', 'x3', 'x6']
  coef=2.00: quant_set=['x1', 'x3', 'x6']
  coef=3.00: quant_set=['x1', 'x6']
  coef=5.00: quant_set=['x1']
  coef=10.00: quant_set=['x1']


In [20]:
SEP = "=" * 90

# DATASET 1: Ebook
print(f"\n{SEP}")
print("  DATASET 1: Ebook — выбор электронной книги (Table 9.1, стр. 355)")
print(SEP)
print_block("SMART",               smart_result)
print_block("SMARTS",              smarts_result)
print_block("TVK CASCADE",         tvk_ebook_result)
print_block("INTERVAL LP",         lp_result)
print_block("MONTE CARLO",         mc_result)
print_block("INTERVAL ANALYTICAL", analytical_result)
print(f"\n  Парето (нормированная матрица): {pareto_result}")

# DATASET 2: Hostel
print(f"\n{SEP}")
print("  DATASET 2: Hostel — выбор хостела (Table 5.5, стр. 171)")
print(SEP)
print_block("SMART HOSTEL",              hostel_smart_result)
print_block("SMARTS HOSTEL",             hostel_smarts_result)
print_block("TVK CASCADE",               tvk_hostel_result)
print_block("INTERVAL LP",               lp_hostel_result)
print_block("MONTE CARLO",               mc_hostel_result)
print_block("INTERVAL ANALYTICAL",       analytical_hostel_result)

# DATASET 3: LP Example 9.2
print(f"\n{SEP}")
print("  DATASET 3: LP Example 9.2 (стр. 350)")
print(SEP)
print_block("LP EXAMPLE 9.2", lp_9_2_result)


  DATASET 1: Ebook — выбор электронной книги (Table 9.1, стр. 355)

SMART
method        : SMART
ranking       : x4 > x3 > x6 > x1 > x2 > x5
weights       : [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
scores        : [0.48571428571428577, 0.41875, 0.5825892857142858, 0.6761904761904762, 0.3416666666666667, 0.5303571428571429]

SMARTS
method        : SMARTS
ranking       : x1 > x6 > x2 > x4 > x3 > x5
weights       : [0.396383, 0.033816, 0.179611, 0.01858, 0.043354, 0.049548, 0.01858, 0.01858, 0.173418, 0.068128]
scores        : [0.660721, 0.514059, 0.346525, 0.452372, 0.285086, 0.535489]

TVK CASCADE
method        : TVK_cascade
pareto_set    : ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
pareto_rank   : x1 = x2 = x3 = x4 = x5 = x6
qual_set      : ['x1', 'x3', 'x4', 'x6']
qual_rank     : x1 = x3 = x4 = x6 > x2 = x5
quant_set     : ['x1', 'x6']
quant_rank    : x1 = x6 > x2 = x3 = x4 = x5

INTERVAL LP
method        : Interval_LP
pareto_lp     : ['x1']
ranking       : x1 > x2 = x6 > x4 > x3 

In [21]:
# DATASET 4: Preflib
print(f"\n{SEP}")
print("  DATASET 4: Preflib — voting rules comparison")
print(SEP)
print_block("SMART",                     preflib_smart_result)
print_block("SMARTS",                    preflib_smarts_result)
print_block("TVK CASCADE",               preflib_tvk_result)
print_block("INTERVAL LP",               preflib_lp_result)
print_block("MONTE CARLO",               preflib_mc_result)
print_block("INTERVAL ANALYTICAL",       analytical_preflib_result)

# ── Матрица Кендалла ──────────────────────────────────────────────
from scipy.stats import kendalltau

def ranking_to_vector(names, ranking_str):
    groups = [[x.strip() for x in g.split("=")] for g in ranking_str.split(">")]
    positions = {}
    pos = 1
    for group in groups:
        avg = pos + (len(group) - 1) / 2
        for name in group:
            positions[name] = avg
        pos += len(group)
    return [positions[n] for n in names]

def kendall_similarity_matrix(names, methods_dict):
    method_names = list(methods_dict.keys())
    vectors = {m: ranking_to_vector(names, r) for m, r in methods_dict.items()}
    n = len(method_names)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            tau, _ = kendalltau(vectors[method_names[i]], vectors[method_names[j]])
            matrix[i, j] = (1 + tau) / 2 * 100
    col_w = 14
    header = f"{'':>{col_w}}" + "".join(f"{m:>{col_w}}" for m in method_names)
    print("\nМатрица сходства ранжировок (Kendall similarity %)")
    print("-" * len(header))
    print(header)
    for i, row_name in enumerate(method_names):
        row = f"{row_name:>{col_w}}" + "".join(f"{matrix[i,j]:>{col_w}.2f}" for j in range(n))
        print(row)
    print("-" * len(header))

print("\nDataset 1 — Ebook:")
kendall_similarity_matrix(ebook_names, {
    "SMART":       smart_result["ranking"],
    "SMARTS":      smarts_result["ranking"],
    "TVK":         tvk_ebook_result["quant_rank"],
    "Interval_LP": lp_result["ranking"],
    "Monte_Carlo": mc_result["ranking"],
    "Analytical":  analytical_result["ranking"],
})

print("\nDataset 2 — Hostel:")
kendall_similarity_matrix(hostel_names, {
    "SMART":       hostel_smart_result["ranking"],
    "SMARTS":      hostel_smarts_result["ranking"],
    "TVK":         tvk_hostel_result["quant_rank"],
    "Interval_LP": lp_hostel_result["ranking"],
    "Monte_Carlo": mc_hostel_result["ranking"],
    "Analytical":  analytical_hostel_result["ranking"],
})

print("\nDataset 4 — Preflib:")
kendall_similarity_matrix(preflib_names, {
    "SMART":       preflib_smart_result["ranking"],
    "SMARTS":      preflib_smarts_result["ranking"],
    "TVK":         preflib_tvk_result["quant_rank"],
    "Interval_LP": preflib_lp_result["ranking"],
    "Monte_Carlo": preflib_mc_result["ranking"],
    "Analytical":  analytical_preflib_result["ranking"],
})


  DATASET 4: Preflib — voting rules comparison

SMART
method        : SMART
ranking       : A (200) > B (209) > C (218) > D (227)
weights       : [0.2, 0.2, 0.2, 0.2, 0.2]
scores        : [0.8, 0.6511946843710678, 0.4913533685102237, 0.05409836065573771]

SMARTS
method        : SMARTS
ranking       : A (200) > B (209) > C (218) > D (227)
weights       : [0.166667, 0.083333, 0.3, 0.2, 0.25]
scores        : [0.75, 0.665598, 0.566464, 0.067623]

TVK CASCADE
method        : TVK_cascade
pareto_set    : ['A (200)', 'B (209)', 'C (218)']
pareto_rank   : A (200) = B (209) = C (218) > D (227)
qual_set      : ['A (200)', 'B (209)', 'C (218)']
qual_rank     : A (200) = B (209) = C (218) > D (227)
quant_set     : ['A (200)']
quant_rank    : A (200) > B (209) = C (218) = D (227)

INTERVAL LP
method        : Interval_LP
pareto_lp     : ['A (200)']
ranking       : A (200) > B (209) > C (218) > D (227)
dominated_by  : {'B (209)': ['A (200)'], 'C (218)': ['A (200)', 'B (209)'], 'D (227)': ['A (200)', 

In [22]:
# Unit test 1: LP solver (Пример 9.2, стр. 350)
assert lp_9_2_result["consistency_feasible"] is True
assert lp_9_2_result["y_P_lambda_z"]         is True
assert lp_9_2_result["y_P_lambda_u"]         is False
print("✅ Unit test 1 (LP Example 9.2) пройден")

# Unit test 2: Аналитический метод — x1 доминирует всех (стр. 356)
assert analytical_result["pareto_set"] == ["x1"], \
    f"Ожидается ['x1'], получено {analytical_result['pareto_set']}"
assert analytical_result["wins"][0] == 5, "x1 должен побеждать всех 5"
print("✅ Unit test 2 (Analytical ebook) пройден")

# Unit test 3: SMARTS hostel — побеждает x3 (стр. 173)
assert hostel_smarts_result["ranking"].startswith("x3"), \
    f"Ожидается x3 первым (стр.173), получено: {hostel_smarts_result['ranking']}"
print("✅ Unit test 3 (SMARTS hostel) пройден")

# Unit test 4: weights_generation суммы
assert abs(sum(W["smarts"]) - 1.0) < 1e-9, "Сумма SMARTS-весов ≠ 1"
print("✅ Unit test 4 (weights_generation) пройден")

# Unit test 5: вложенность
tvk_result = run_tvk_cascade(ebook_names, ebook_grade_matrix,
                              ranks=ebook_ranks, importance_coef=3.0)
assert set(tvk_result["qual_set"]).issubset(set(tvk_result["pareto_set"])), \
    "TVK_qual должно быть подмножеством Pareto"
assert set(tvk_result["quant_set"]).issubset(set(tvk_result["qual_set"])), \
    "TVK_quant должно быть подмножеством TVK_qual"
assert "x1" in tvk_result["quant_set"], \
    f"x1 должна быть в quant_set при coef=3.0, получено: {tvk_result['quant_set']}"
print("✅ Unit test 5 (вложенность TVK + x1 в quant_set) пройден")

# Unit test 6: вложенность на hostel
tvk_hostel = run_tvk_cascade(hostel_names, hostel_grade_matrix, ranks=hostel_ranks)
assert set(tvk_hostel["qual_set"]).issubset(set(tvk_hostel["pareto_set"]))
assert set(tvk_hostel["quant_set"]).issubset(set(tvk_hostel["qual_set"]))
assert "x3" in tvk_hostel["pareto_set"], "x3 должен быть в Парето хостела"
print("✅ Unit test 6 (вложенность TVK hostel) пройден")

print("\n✅ Все unit tests пройдены")

✅ Unit test 1 (LP Example 9.2) пройден
✅ Unit test 2 (Analytical ebook) пройден
✅ Unit test 3 (SMARTS hostel) пройден
✅ Unit test 4 (weights_generation) пройден
✅ Unit test 5 (вложенность TVK + x1 в quant_set) пройден
✅ Unit test 6 (вложенность TVK hostel) пройден

✅ Все unit tests пройдены
